# Сбор данных с Web-scraping и API для гуманитарных исследований
## Блок 2. Веб-страницы
## Занятие 3. HTML, BeautifulSoup и XML

На занятиях 1 и 2 данные приходили от API в формате JSON, то есть уже разложенными по ключам словаря. Но многие сведения, нужные гуманитарию, существуют только в виде веб-страницы, написанной для человека: списка, таблицы, абзаца текста. Чтобы получить из такой страницы таблицу, её разметку нужно разобрать. На этом занятии разбирается, как устроен язык разметки HTML, как библиотека BeautifulSoup превращает страницу в дерево, по которому можно искать нужные элементы, и как устроен XML — родственный HTML формат, в котором хранятся многие гуманитарные данные, в том числе размеченные тексты.

**Материал:** статья «Библиография Антона Чехова» в русской Википедии, полученная через MediaWiki API; ответ того же API в формате XML; ⭐ пьесы Чехова в разметке TEI из корпуса DraCor.

**Исследовательская задача.** Как менялось творчество Чехова по годам? Когда он писал больше всего рассказов, когда перешёл к повестям и пьесам и какие рассказы стали «заметными», то есть получили отдельную статью в Википедии? Задача продолжает занятие 2: там выяснилось, что годы в каталоге Open Library говорят о каталоге, а не о Чехове. Теперь нужен источник, в котором годы относятся к самим произведениям. К концу занятия у вас будет таблица из примерно 650 произведений с годами и разделами, и по ней можно будет ответить на вопрос, учитывая, кем и как составлен список.

**Путь занятия:**

1. повторение: запрос к API, регулярные выражения;
2. задача, которую API не решает: нужные данные есть только в разметке статьи;
3. HTML: с нуля для начинающих и кратко для продолжающих;
4. BeautifulSoup и CSS-селекторы: сначала на учебном фрагменте, затем на статье;
5. превращение строки списка в поля таблицы;
6. проверка набора данных и ответ на вопрос;
7. XML: с нуля для начинающих; ⭐ TEI и корпус пьес DraCor для продолжающих.

---

Ячейки запускаются сверху вниз, потому что каждая часть опирается на предыдущую. Задания для самостоятельной работы отмечены значком ✏️; код пишется в ячейке под заданием на месте комментария `# ВАШ КОД`. Остальные значки означают следующее:

- 🔁 — блоки повторения. Они освежают то, что изучалось на занятиях 1–2 и в курсе «Цифровые технологии в профессиональной деятельности».
- 🌱 — подробные объяснения с нуля для тех, кто встречается с темой впервые. Продолжающие могут их свернуть и перейти к краткому напоминанию, отмеченному ⭐.
- 🖐 — мини-задания на одну-две минуты сразу после объяснения: в них нужно руками попробовать только что показанное. Они выполняются без ИИ и никуда не отправляются.

- 📨 — задания предполагают ваш выбор и рассуждение. Их результат отправляется в чат курса, чтобы его можно было обсудить вместе.
- ⭐ — части и задания продвинутого уровня. На базовом уровне их можно пропустить: остальная тетрадка от них не зависит.

У каждого задания есть значок, который показывает, можно ли пользоваться ИИ-ассистентом:

- 🚫 — без ИИ;
- 💬 — ассистента можно спрашивать только об ошибках и непонятных местах, а код вы пишете сами;
- 🤖 — ИИ разрешён; в журнал `AI_LOG.md` записывается, что вы спрашивали, что приняли из ответа и что исправили.

С этого занятия начинается второй блок курса, и впервые появляются задания с 🤖.

---
## Часть 0. Подготовка

Библиотека BeautifulSoup устанавливается вместе с остальными зависимостями курса из файла `requirements.txt` (пакет `beautifulsoup4`), а импортируется под именем `bs4`. Пакет `lxml` нужен для разбора XML в части 7, пакет `matplotlib` — для графика в части 6.

In [ ]:
import json
import re
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import HTML, display

In [ ]:
WIKI_API = "https://ru.wikipedia.org/w/api.php"

# Без осмысленного User-Agent Википедия отвечает кодом 403 (занятие 1)
HEADERS = {
    "User-Agent": "miniDH-scraping-course/1.0 (https://github.com/Minidegree-DH/miniDH-scraping)"
}

---
## Часть 1. 🔁 Повторение

### Запрос к API

На занятии 2 была написана функция `get_json()`, которая отправляет запрос, повторяет его при временных сбоях и возвращает словарь. Она понадобится и здесь.

In [ ]:
def get_json(url: str, params: dict, attempts: int = 3, headers: dict = HEADERS) -> dict:
    """Отправляет GET-запрос и возвращает JSON; при временных сбоях повторяет запрос."""
    for attempt in range(1, attempts + 1):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=10)
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            print(f"Попытка {attempt}: нет ответа от сервера")
            time.sleep(5)
            continue

        if response.status_code in (429, 503):
            print(f"Попытка {attempt}: код {response.status_code}, сервер просит подождать")
            time.sleep(5)
            continue

        response.raise_for_status()
        return response.json()

    raise RuntimeError(f"Не удалось получить ответ за {attempts} попытки: {url}")

### Регулярные выражения

Регулярные выражения встречались в курсе «Цифровые технологии в профессиональной деятельности». Они понадобятся в части 5, чтобы выделить из строки списка год и название произведения. Напомним основные обозначения:

| Запись | Что означает |
|---|---|
| `\d` | одна цифра; `\d{4}` — ровно четыре цифры |
| `^` и `$` | начало и конец строки |
| `( … )` | группа: часть совпадения, которую можно получить отдельно |
| `.*` | любые символы в любом количестве |

Функция `re.match(шаблон, строка)` ищет совпадение с начала строки и возвращает объект совпадения или `None`. Метод `.group(1)` возвращает содержимое первой группы, `.groups()` — всех групп сразу.

In [ ]:
line = "1888 — Степь"

match = re.match(r"(\d{4}) — (.*)", line)

print(match.group(1))   # первая группа: год
print(match.group(2))   # вторая группа: название

In [ ]:
# Если строка не подходит под шаблон, re.match возвращает None
print(re.match(r"(\d{4}) — (.*)", "Записные книжки"))

#### ✏️🚫 Задание 1

Выделите год из каждой строки списка `lines` и выведите строку и год. Если года нет, выведите `None`.

In [ ]:
lines = ["1884 — Хамелеон", "1886 — Тоска", "Записные книжки", "1899 — Дама с собачкой"]

# ВАШ КОД

#### ✏️🚫 Задание 2

В библиографии Чехова встречаются строки с диапазоном лет, например «1893—1895 — Остров Сахалин». Напишите шаблон с тремя группами: год начала, год окончания и название. Проверьте его на строке `"1893—1895 — Остров Сахалин"` и выведите результат метода `.groups()`.

*Подсказка:* между годами стоит длинное тире `—` без пробелов, а перед названием — тире с пробелами.

In [ ]:
line = "1893—1895 — Остров Сахалин"

# ВАШ КОД

---
## Часть 2. Задача, которую API не решает

### Что даёт API

На занятии 2 статьи категории собирались функцией `get_all_category_members()`. В русской Википедии есть «Категория:Рассказы Антона Чехова»; все её статьи помещаются в один ответ, поэтому здесь достаточно одного запроса.

In [ ]:
params = {
    "action": "query",
    "list": "categorymembers",
    "cmtitle": "Категория:Рассказы Антона Чехова",
    "cmtype": "page",
    "cmlimit": 500,
    "format": "json",
}

stories_category = get_json(WIKI_API, params)["query"]["categorymembers"]

len(stories_category)

В категории около 190 статей. Это рассказы, о которых в Википедии написаны отдельные статьи, но Чехов написал несколько сотен рассказов. Кроме того, у статей категории нет годов: чтобы узнать год каждого рассказа, пришлось бы загружать и разбирать каждую статью.

### Где лежат нужные данные

В Википедии есть статья [«Библиография Антона Чехова»](https://ru.wikipedia.org/wiki/Библиография_Антона_Чехова). Откройте её в браузере: это список произведений, разбитый на разделы (пьесы, повести, рассказы, фельетоны и др.), и в каждой строке указаны год и название. Таких строк около 650.

У API нет запроса, который вернул бы этот список как набор записей: список — часть текста статьи. Но API может вернуть текст статьи в виде HTML, то есть той разметки, которую видит браузер. Для этого служит действие `parse`. Так соблюдается правило курса «API приоритетнее скрейпинга»: страница загружается через API, по его правилам и с его лимитами, а разбирать её содержимое приходится уже самим.

### Параметры действия `parse`

Параметры описаны на странице справки [API:Parsing wikitext](https://www.mediawiki.org/wiki/API:Parsing_wikitext); их можно получить и знакомым по занятию 2 способом, через `action=paraminfo` с `modules=parse`.

| Параметр | Значение в запросе | Что задаёт | Другие варианты |
|---|---|---|---|
| `action` | `parse` | разобрать страницу и вернуть результат | `query` (занятия 1–2) |
| `page` | `Библиография Антона Чехова` | какую страницу разобрать | `pageid` — номер страницы |
| `prop` | `text\|revid` | что вернуть: HTML-текст и номер версии страницы | `sections` (оглавление), `links` (все ссылки), `categories` (категории) и другие |
| `formatversion` | `2` | HTML приходит обычной строкой | при `formatversion=1` текст лежит в словаре под ключом `*` |
| `format` | `json` | формат ответа | `xml` (часть 7) |

Есть и полезные необязательные параметры. `disableeditsection=1` убирает из HTML ссылки «[править]» возле заголовков; `section=3` возвращает только третий раздел статьи. Здесь они не используются, чтобы увидеть страницу такой, какая она есть.

In [ ]:
params = {
    "action": "parse",
    "page": "Библиография Антона Чехова",
    "prop": "text|revid",
    "formatversion": 2,
    "format": "json",
}

data = get_json(WIKI_API, params)

data["parse"].keys()

🖐 **Попробуйте.** Измените в запросе один параметр: `prop="sections"` вместо `"text|revid"`. В ответе `data["parse"]["sections"]` окажется оглавление статьи. Выведите номер (`number`) и название (`line`) каждого раздела.

In [ ]:
params_sections = dict(params)
# ВАШ КОД

In [ ]:
# Номер версии страницы: статьи Википедии постоянно правятся,
# и для воспроизводимости важно записать, по какой версии собран набор
revid = data["parse"]["revid"]
revid

In [ ]:
# HTML-текст статьи — одна длинная строка
html = data["parse"]["text"]

print(type(html))
print(len(html))

In [ ]:
# Первые 1000 символов
print(html[:1000])

🖐 **Попробуйте.** Строковыми методами из занятия 1 посчитайте, сколько раз в строке `html` встречаются открывающие теги строк списка (`"<li"`) и заголовков разделов (`"<h2"`). Совпадает ли число строк с примерно 650 произведениями из статьи? Если нет, предположите, откуда лишние.

In [ ]:
# ВАШ КОД

Разметку можно прочитать глазами, но с трудом: в строке перемешаны текст, служебные знаки в угловых скобках и атрибуты. Строковыми методами из занятия 1 (`.find()`, `.count()`) в ней можно найти отдельные слова, но нельзя надёжно отделить строку списка от ссылки в сноске или от служебной вставки. Чтобы с разметкой можно было работать, нужно сначала понять, как она устроена (часть 3), а затем воспользоваться библиотекой, которая читает её за нас (часть 4).

---
## Часть 3. HTML

Эта часть состоит из двух вариантов. Подраздел 🌱 рассказывает об HTML с нуля и предлагает собрать собственную веб-страницу; он рассчитан на тех, кто с HTML не работал. Подраздел ⭐ — краткое напоминание для тех, кто HTML знает; после него можно сразу переходить к части 4.

### 🌱 HTML с нуля

#### Что такое разметка

**HTML** (HyperText Markup Language, язык гипертекстовой разметки) — язык, на котором написаны веб-страницы. Файл HTML — это обычный текст, в который вставлены указания для браузера: что здесь заголовок, что абзац, что список, что ссылка. Браузер читает эти указания и показывает текст оформленным, а сами указания на экране не видны.

Указания записываются **тегами** — словами в угловых скобках. Большинство тегов парные: **открывающий** тег `<p>` отмечает начало абзаца, **закрывающий** тег `</p>` с косой чертой — его конец. Открывающий тег, закрывающий тег и всё, что между ними, вместе называются **элементом**:

```
<p>Пьеса почти готова.</p>
 │         │         │
 │         │         └─ закрывающий тег
 │         └─────────── содержимое элемента
 └───────────────────── открывающий тег
```

Некоторые элементы не имеют содержимого и закрывающего тега: например, `<br>` (перевод строки) или `<img>` (изображение). Их называют пустыми.

#### Основные теги

Имена тегов — сокращения английских слов. Если помнить, от чего образовано имя, тег легко узнать в чужой разметке.

| Тег | Расшифровка | Что размечает |
|---|---|---|
| `<h1>` … `<h6>` | heading, заголовок | заголовки шести уровней: `<h1>` — главный, `<h2>` — раздел, `<h3>` — подраздел |
| `<p>` | paragraph, абзац | абзац текста |
| `<ul>` | unordered list, неупорядоченный список | маркированный список (с точками) |
| `<ol>` | ordered list, упорядоченный список | нумерованный список |
| `<li>` | list item, элемент списка | одна строка списка внутри `<ul>` или `<ol>` |
| `<a>` | anchor, якорь | ссылка; адрес — в атрибуте `href` (hypertext reference, гипертекстовая ссылка) |
| `<strong>`, `<em>` | strong, сильный; emphasis, выделение | важный и выделенный текст (обычно полужирный и курсив) |
| `<table>`, `<tr>` | table, таблица; table row, строка таблицы | таблица и её строка |
| `<th>`, `<td>` | table header, заголовок таблицы; table data, данные таблицы | ячейка заголовка и обычная ячейка |
| `<div>` | division, раздел | блок, контейнер для других элементов, без собственного смысла |
| `<span>` | span, отрезок | фрагмент внутри строки, обычно для оформления |
| `<img>` | image, изображение | картинка; адрес файла — в атрибуте `src` (source, источник) |
| `<br>` | break, разрыв | перевод строки |

🖐 **Попробуйте.** Ниже — фрагмент без разметки. Функция `display(HTML(строка))` показывает разметку так, как её отобразит браузер (подробнее о ней — в конце подраздела). Запустите ячейку, затем разметьте фрагмент: сделайте первую строку заголовком `<h2>`, три книги — нумерованным списком, а одно слово в последней фразе выделите тегом `<em>`. Запустите ячейку ещё раз и сравните.

In [ ]:
fragment = """
Три пьесы Чехова
Чайка
Дядя Ваня
Три сестры
Все три пьесы впервые поставил Московский Художественный театр.
"""

display(HTML(fragment))

#### Атрибуты

В открывающем теге могут стоять **атрибуты** — дополнительные сведения об элементе в виде пар `имя="значение"`:

```html
<a href="https://ru.wikipedia.org/wiki/Степь_(повесть)" title="Степь (повесть)">Степь</a>
```

Здесь у ссылки два атрибута: `href` — куда она ведёт, `title` — подсказка, которая появляется при наведении мыши. Два атрибута встречаются почти у любых элементов:

- `id` (identifier, идентификатор) — уникальное имя элемента на странице: `<h2 id="Пьесы">`;
- `class` (класс) — один или несколько классов через пробел: `<li class="play famous">`. Классы задают оформление, но для скрейпинга они тоже полезны: по ним удобно находить элементы одного вида.

🖐 **Попробуйте.** Превратите название «Чайка» в ссылку на статью Википедии `https://ru.wikipedia.org/wiki/Чайка_(пьеса)` с подсказкой `title="Чайка (пьеса)"`, а строке «Три сестры» добавьте `class="mhat"`. Наведите мышь на ссылку в результате: должна появиться подсказка.

In [ ]:
fragment = """
<ol>
  <li>Чайка</li>
  <li>Дядя Ваня</li>
  <li>Три сестры</li>
</ol>
"""

# ВАШ КОД: измените fragment и покажите его

#### Вложенность и дерево

Элементы вкладываются друг в друга: список содержит строки, строка — ссылку, ссылка — текст. Вложенные элементы должны закрываться в обратном порядке: `<li><a>…</a></li>`, а не `<li><a>…</li></a>`. Поэтому любую страницу можно нарисовать как **дерево**, в котором у каждого элемента есть **родитель** (элемент, в который он вложен), а у многих — **потомки** (вложенные в него элементы) и **соседи** (элементы с тем же родителем):

```
div
├── h2  «Пьесы»
└── ol
    ├── li  «1896 — Чайка»
    │   └── a  «Чайка»
    └── li  «1904 — Вишнёвый сад»
        └── a  «Вишнёвый сад»
```

По такому дереву и будет вестись поиск в части 4: «найти все `li` внутри `ol`, который идёт после заголовка "Пьесы"».

🖐 **Попробуйте.** Нарисуйте в ячейке ниже дерево для этого фрагмента так же, как на схеме выше, и ответьте: кто родитель элемента `<em>`? Кто соседи у первого `<li>`?

```html
<div>
  <p>Пьесы <em>Чехова</em>:</p>
  <ul>
    <li>Чайка</li>
    <li>Иванов</li>
  </ul>
</div>
```

*Дерево:*

*Ответ:*

#### Каркас документа

Полноценная страница начинается с объявления типа документа и состоит из двух частей: заголовочной `<head>` со служебными сведениями и основной `<body>` с тем, что видно на экране.

```html
<!DOCTYPE html>
<html lang="ru">
  <head>
    <meta charset="utf-8">
    <title>Название во вкладке браузера</title>
  </head>
  <body>
    <h1>Заголовок страницы</h1>
    <p>Первый абзац.</p>
  </body>
</html>
```

Строка `<meta charset="utf-8">` сообщает браузеру кодировку, без неё кириллица может отобразиться неправильно. Отступы в HTML ни на что не влияют и нужны только для удобства чтения.

Несколько знаков в HTML служебные, поэтому для них есть особые записи — **сущности**: `&lt;` и `&gt;` (less than, greater than) обозначают знаки `<` и `>`, `&amp;` (ampersand) — знак `&`, а `&nbsp;` (non-breaking space) — неразрывный пробел, который не даёт браузеру перенести строку в этом месте. Неразрывный пробел ещё встретится в части 4.

🖐 **Попробуйте.** Запустите ячейку: в показанном тексте пропало слово `<p>`, потому что браузер принял его за тег. Исправьте строку с помощью сущностей так, чтобы фраза отображалась целиком. Заодно соедините «А. П.» и «Чехов» неразрывными пробелами, чтобы инициалы не отрывались от фамилии при переносе.

In [ ]:
text_html = "Тег <p> у А. П. Чехова не встречается: это тег HTML."

display(HTML(text_html))

#### Как посмотреть свою страницу

HTML-документ можно показать прямо в тетрадке функцией `display(HTML(строка))` или сохранить в файл с расширением `.html` и открыть в браузере двойным щелчком.

In [ ]:
page = """
<!DOCTYPE html>
<html lang="ru">
  <head>
    <meta charset="utf-8">
    <title>Пробная страница</title>
  </head>
  <body>
    <h1>Чехов-драматург</h1>
    <p>Четыре большие пьесы <strong>Антона Чехова</strong>:</p>
    <ol>
      <li>Чайка</li>
      <li>Дядя Ваня</li>
      <li>Три сестры</li>
      <li>Вишнёвый сад</li>
    </ol>
  </body>
</html>
"""

# Показ прямо в тетрадке
display(HTML(page))

In [ ]:
# Сохранение в файл: его можно открыть в браузере
Path("trial_page.html").write_text(page, encoding="utf-8")

#### ✏️🚫 Задание 3

Соберите собственную HTML-страницу о писателе, художнике или явлении из вашей области. На странице должны быть:

1. каркас документа с кодировкой и названием во вкладке (`<title>`);
2. заголовок `<h1>` и абзац `<p>` с коротким описанием, в котором одно слово выделено тегом `<strong>`;
3. нумерованный список из трёх-четырёх произведений, где хотя бы два пункта — ссылки `<a>` на статьи Википедии с атрибутами `href` и `title`;
4. у одной строки списка атрибут `class="important"`.

Покажите страницу в тетрадке, сохраните её в файл `my_page.html` и откройте файл в браузере.

In [ ]:
my_page = """
<!DOCTYPE html>
<html lang="ru">

"""

# ВАШ КОД: показ в тетрадке и сохранение в my_page.html

#### ✏️🚫 Задание 4

Добавьте на свою страницу таблицу из трёх строк с данными о тех же произведениях: столбцы «Год», «Название», «Жанр». Первая строка таблицы — заголовки (`<th>`), остальные — данные (`<td>`). Обновите файл и проверьте, как таблица выглядит в браузере. Если таблица отображается без рамок, добавьте к тегу `<table>` атрибут `border="1"`.

In [ ]:
# ВАШ КОД

### ⭐ HTML: краткое напоминание

Для тех, кто с HTML знаком, достаточно напомнить то, что понадобится при разборе статьи Википедии:

- страница — дерево элементов; элемент — открывающий тег, содержимое и закрывающий тег; у элемента есть родитель, потомки и соседи;
- `id` уникален на странице, `class` — список классов через пробел; по классам удобно находить элементы одного вида;
- в Википедии заголовок раздела `<h2>` вложен в блок `<div class="mw-heading">`, а следом за этим блоком идёт список `<ol>` или `<ul>` со строками `<li>`;
- в тексте встречаются сущности, прежде всего неразрывный пробел `&nbsp;`, который в Python превращается в символ `\xa0`.

Можно переходить к части 4.

### Инструменты разработчика

Как узнать, какими тегами размечена нужная часть настоящей страницы? В любом современном браузере есть **инструменты разработчика**: если щёлкнуть правой кнопкой мыши по элементу страницы и выбрать «Просмотреть код» (Inspect), откроется панель с деревом HTML, где выбранный элемент подсвечен.

На занятии преподаватель покажет в инструментах разработчика статью «Библиография Антона Чехова»: как размечены заголовок раздела и строка списка, где лежат служебные ссылки «править» и чем отличается строка, в которой год оформлен ссылкой. После занятия эти шаги полезно повторить самостоятельно.

---
## Часть 4. BeautifulSoup и CSS-селекторы

### Зачем нужна библиотека

HTML-страница, полученная из API, — это строка. Искать в строке элементы неудобно: тег может стоять с атрибутами или без, элементы бывают вложены друг в друга, а одинаковые теги встречаются в разных местах страницы. Нужна программа, которая прочитает строку и восстановит из неё дерево элементов. Такую программу называют **парсером** (от англ. parse — разбирать).

**BeautifulSoup** — библиотека Python, которая берёт HTML-строку, передаёт её парсеру и возвращает дерево в виде объектов Python. С этим деревом можно работать так же, как со страницей в инструментах разработчика: найти элемент, прочитать его текст и атрибуты, перейти к родителю, потомкам или соседям.

Функция `BeautifulSoup(строка, парсер)` принимает два аргумента. Второй выбирает парсер:

| Парсер | Что это | Когда использовать |
|---|---|---|
| `"html.parser"` | встроенный в Python, устанавливать не нужно | для учебных и большинства обычных страниц |
| `"lxml"` | внешний пакет, быстрее и терпимее к ошибкам разметки | для больших страниц и страниц с ошибками |
| `"xml"` | парсер XML на основе `lxml` | для XML-документов (часть 7) |

Результат разбора состоит из объектов двух основных видов. **Тег** (`Tag`) — элемент дерева: у него есть имя (`.name`), атрибуты и потомки. **Строка** (`NavigableString`) — кусок текста внутри элемента.

### Основные операции

| Операция | Что делает | Что возвращает |
|---|---|---|
| `soup.find("li")` | ищет первый элемент с тегом `li` | тег или `None`, если ничего не найдено |
| `soup.find_all("li")` | ищет все элементы с тегом `li` | список тегов (может быть пустым) |
| `tag.get_text()` | собирает весь текст внутри элемента, без тегов | строку |
| `tag["href"]` | значение атрибута `href` | строку; если атрибута нет — ошибка `KeyError` |
| `tag.get("href")` | то же, но без ошибки | строку или `None` |
| `tag.parent` | родитель элемента | тег |
| `tag.children` | прямые потомки элемента по порядку | последовательность тегов и строк |
| `tag.find_next_sibling("ol")` | следующий сосед с тегом `ol` | тег или `None` |

Методы `find` и `find_all` можно вызывать не только у всего дерева, но и у любого тега: тогда поиск идёт только внутри этого тега. У них есть полезные аргументы: `class_="имя"` ищет элементы с классом (подчёркивание нужно потому, что `class` — служебное слово Python), `id="имя"` — элемент с идентификатором, `recursive=False` ограничивает поиск прямыми потомками.

### Учебный фрагмент

Сначала операции проверяются на маленьком фрагменте, устроенном так же, как статья-библиография: два раздела, в каждом по два произведения.

In [ ]:
snippet = """
<div class="bibliography">
  <h2 id="plays">Пьесы</h2>
  <ol>
    <li>1896 — <a href="/wiki/Чайка_(пьеса)" title="Чайка (пьеса)">Чайка</a></li>
    <li>1904 — <a href="/wiki/Вишнёвый_сад" title="Вишнёвый сад">Вишнёвый сад</a></li>
  </ol>
  <h2 id="novellas">Повести</h2>
  <ol>
    <li class="famous">1888 — <a href="/wiki/Степь_(повесть)" title="Степь (повесть)">Степь</a></li>
    <li>1882 — Ненужная победа</li>
  </ol>
</div>
"""

soup = BeautifulSoup(snippet, "html.parser")

type(soup)

In [ ]:
# Первый элемент с тегом h2: тег целиком, вместе с разметкой
soup.find("h2")

In [ ]:
# Текст внутри элемента, без тегов
soup.find("h2").get_text()

In [ ]:
# Все элементы с тегом li — список
items = soup.find_all("li")

len(items)

In [ ]:
# Первая строка списка: разметка и текст
print(items[0])
print(items[0].get_text())

In [ ]:
# Ссылка внутри первой строки; поиск идёт только внутри этого тега
link = items[0].find("a")

link

In [ ]:
# Атрибуты читаются как ключи словаря
print(link["href"])
print(link.get("title"))

In [ ]:
# В последней строке ссылки нет: find возвращает None
print(items[3].find("a"))

🖐 **Попробуйте.** Выведите текст второго заголовка `h2` фрагмента и список подсказок `title` всех ссылок фрагмента.

In [ ]:
# ВАШ КОД

In [ ]:
# Поиск по атрибутам: заголовок с id="novellas" и строка с классом famous
print(soup.find("h2", id="novellas").get_text())
print(soup.find("li", class_="famous").get_text())

In [ ]:
# Родитель строки — список ol; сосед заголовка справа — тоже список
print(items[0].parent.name)
print(soup.find("h2", id="novellas").find_next_sibling("ol"))

🖐 **Попробуйте.** Начните со ссылки «Степь» (`soup.find("a", title="Степь (повесть)")`) и, поднимаясь по родителям, выведите: текст строки, в которой стоит ссылка, имя тега её списка и имя тега, в который вложен список.

In [ ]:
steppe = soup.find("a", title="Степь (повесть)")

# ВАШ КОД

#### ✏️🚫 Задание 5

Для каждой строки учебного фрагмента выведите текст строки, адрес ссылки и подсказку `title`; если ссылки нет, выведите `None`.

In [ ]:
# ВАШ КОД

#### ✏️🚫 Задание 6

Прочитайте свою страницу из файла `my_page.html` (задание 3), разберите её BeautifulSoup и выведите: текст заголовка `<h1>`, тексты всех строк списка, адреса всех ссылок и текст строки с классом `important`. Если задание 3 вы пропустили, разберите строку `page` из подраздела 🌱.

In [ ]:
# ВАШ КОД

### CSS-селекторы

Кроме `find` и `find_all`, в BeautifulSoup есть второй способ поиска — **CSS-селекторы**. Чтобы понять, откуда они взялись, нужно знать, что такое CSS.

**CSS** (Cascading Style Sheets, каскадные таблицы стилей) — язык, на котором описывается оформление веб-страниц: цвета, шрифты, отступы. Правило CSS состоит из **селектора**, который говорит, к каким элементам правило относится, и из списка свойств:

```css
li.famous { color: darkred; }
```

Это правило означает: «строки списка с классом `famous` показать тёмно-красным». Часть `li.famous` и есть селектор — короткая запись условия, по которому выбираются элементы. Разработчики браузеров и веб-страниц пользуются селекторами каждый день, поэтому язык селекторов хорошо развит и подходит не только для оформления, но и для поиска элементов при скрейпинге.

| Селектор | Что выбирает | То же через `find_all` |
|---|---|---|
| `li` | все элементы `li` | `find_all("li")` |
| `.famous` | все элементы с классом `famous` | `find_all(class_="famous")` |
| `#plays` | элемент с `id="plays"` | `find(id="plays")` |
| `ol li` | элементы `li` где угодно внутри `ol` (пробел — «внутри») | цикл по `ol` и `find_all("li")` в каждом |
| `ol > li` | элементы `li`, которые являются прямыми потомками `ol` | `find_all("li", recursive=False)` в каждом `ol` |
| `h2 + ol` | список `ol`, который идёт сразу после заголовка `h2` | `find_next_sibling("ol")` |
| `a[title]` | ссылки, у которых есть атрибут `title` | `find_all("a", title=True)` |
| `h2, h3` | заголовки обоих уровней (запятая — «или») | `find_all(["h2", "h3"])` |

Метод `.select(селектор)` возвращает список подходящих элементов, метод `.select_one(селектор)` — первый из них или `None`. Селекторы удобнее `find_all`, когда условие состоит из нескольких уровней вложенности; `find` и `find_all` нагляднее, когда условие простое.

🖐 **Попробуйте.** Селектор CSS можно увидеть в действии. В ячейке ниже правило оформления (в теге `<style>`) окрашивает строки с классом `famous`. Запустите ячейку, затем по очереди замените селектор `.demo li.famous` на `.demo #plays + ol li` и на `.demo a` и посмотрите, какие элементы окрашиваются. Приставка `.demo` ограничивает правило учебным фрагментом, чтобы оно не перекрасило саму тетрадку.

In [ ]:
style = "<style> .demo li.famous { color: darkred; } </style>"

display(HTML(style + '<div class="demo">' + snippet + "</div>"))

In [ ]:
# Все строки списков учебного фрагмента
soup.select("ol > li")

In [ ]:
# Ссылки с атрибутом title; для каждой выводим подсказку
[link["title"] for link in soup.select("a[title]")]

In [ ]:
# Строки списка, который идёт сразу после заголовка с id="novellas"
soup.select("#novellas + ol > li")

Селектор нужного элемента можно скопировать прямо из инструментов разработчика: правая кнопка мыши на элементе дерева → «Копировать» → «Копировать селектор». Такие селекторы часто получаются длинными и привязанными к случайным подробностям разметки, поэтому их стоит сокращать до нужного условия.

#### ✏️🚫 Задание 7

Напишите CSS-селекторы для учебного фрагмента и выведите результат каждого:

1. все ссылки внутри строк списков;
2. строка с классом `famous`;
3. строки раздела «Пьесы» (у заголовка `id="plays"`).

In [ ]:
# ВАШ КОД

### BeautifulSoup на настоящей странице

Теперь те же приёмы переносятся на статью «Библиография Антона Чехова». Всё содержимое статьи лежит в блоке `<div class="mw-parser-output">`.

In [ ]:
soup = BeautifulSoup(html, "html.parser")

root = soup.find("div", class_="mw-parser-output")

root.name

In [ ]:
# Заголовки разделов второго уровня
for heading in root.find_all("h2"):
    print(heading.get_text())

🖐 **Попробуйте.** В статье есть и подразделы третьего уровня. Выведите тексты всех заголовков `h3` и посчитайте их.

In [ ]:
# ВАШ КОД

В заголовках нет служебных слов «править»: ссылки на правку лежат не внутри `<h2>`, а рядом с ним, в том же блоке `<div class="mw-heading">`. Поэтому список раздела ищется так: заголовок → его родитель (блок `div.mw-heading`) → следующий за блоком список.

In [ ]:
# Заголовок раздела «Пьесы»: у h2 есть атрибут id
plays_heading = root.find("h2", id="Пьесы")

# Родитель заголовка — блок div.mw-heading; список идёт следом за этим блоком
plays_list = plays_heading.parent.find_next_sibling("ol")

# recursive=False: только строки этого списка, без вложенных списков
plays_items = plays_list.find_all("li", recursive=False)

len(plays_items)

In [ ]:
# Первая строка раздела: разметка и текст
print(plays_items[0])
print(plays_items[0].get_text())

In [ ]:
# repr показывает невидимые символы: между годом и тире стоит неразрывный пробел \xa0
repr(plays_items[0].get_text())

Между годом и тире стоит **неразрывный пробел** (`&nbsp;` в HTML, `\xa0` в Python). На экране он не отличается от обычного, но шаблон регулярного выражения с обычным пробелом на нём не сработает. Поэтому перед разбором текст нормализуется: `text.replace("\xa0", " ")`.

Те же строки можно найти и селектором. В нём учтено, что заголовок вложен в блок `div.mw-heading`, а список — сосед этого блока; запись `:has(...)` означает «элемент, внутри которого есть подходящий потомок».

In [ ]:
plays_by_selector = root.select("div.mw-heading:has(h2#Пьесы) + ol > li")

len(plays_by_selector)

#### ✏️💬 Задание 8

Найдите все строки раздела «Повести» любым из двух способов и выведите их текст, заменив неразрывный пробел обычным. Сколько повестей перечислено?

In [ ]:
# ВАШ КОД

### Обход страницы по порядку

Чтобы собрать все разделы сразу, страницу удобно обойти сверху вниз и запоминать, какой раздел сейчас идёт. Атрибут `root.children` перебирает прямых потомков блока по порядку. Среди потомков встречаются не только теги, но и куски текста между ними (обычно переводы строк); у таких кусков атрибут `.name` равен `None`, и их нужно пропускать.

Порядок действий для каждого потомка:

1. если это блок заголовка (`div` с классом `mw-heading`), запомнить текст заголовка как текущий раздел;
2. если это список (`ol` или `ul`), пройти по его строкам верхнего уровня и для каждой записать пару «раздел — строка»;
3. остальное пропустить.

Классы элемента возвращаются списком: `element.get("class", [])`. Значение по умолчанию — пустой список, потому что у части элементов атрибута `class` нет.

#### ✏️💬 Задание 9

Допишите цикл по трём шагам выше: соберите в список `pairs` пары `(раздел, строка li)` для всех разделов, кроме «См. также» и «Ссылки». Выведите число пар и число строк в каждом разделе.

*Подсказка:* заголовок внутри блока может быть `h2` или `h3`; `find(["h2", "h3"])` ищет любой из двух тегов.

In [ ]:
SKIP = ["См. также", "Ссылки"]

pairs = []
section = None

for element in root.children:
    # ВАШ КОД
    ...

len(pairs)

---
## Часть 5. Из строки — поля таблицы

Каждая строка списка содержит год (иногда диапазон лет), название и, возможно, ссылку на статью о произведении. Нужно разложить строку на поля:

| Поле | Откуда берётся |
|---|---|
| `section` | раздел, в котором стоит строка |
| `year`, `year_end` | первая и вторая группы регулярного выражения |
| `title` | третья группа: текст после тире |
| `has_article` | есть ли в строке ссылка на статью о произведении |
| `article` | подсказка `title` этой ссылки, то есть название статьи |
| `text` | полный текст строки, чтобы можно было проверить разбор |

Две детали требуют внимания. Во-первых, в некоторых строках год сам оформлен ссылкой на статью вида «1882 год в литературе», и такая ссылка — не ссылка на произведение. Во-вторых, в строках бывают сноски (тег `<sup>`, от superscript — надстрочный) и вложенные списки, которые не должны попасть в текст строки. Метод `.decompose()` удаляет элемент из дерева вместе с содержимым.

Сначала разбор проверяется на одной строке.

In [ ]:
section, item = pairs[0]

# Удаляем сноски и вложенные списки; decompose изменяет дерево
for extra in item.find_all(["sup", "ul", "ol"]):
    extra.decompose()

text = item.get_text().replace("\xa0", " ").strip()
text

In [ ]:
# Год, необязательный второй год и название.
# (?: … )? — необязательная группа, которая не попадает в результат
pattern = r"^(\d{4})(?:—(\d{4}))? — (.*)$"

match = re.match(pattern, text)
match.groups()

In [ ]:
# Ссылки строки, кроме ссылок вида «1882 год в литературе»
links = [link for link in item.find_all("a") if not link.get("title", "").endswith("год в литературе")]

links

🖐 **Попробуйте.** Повторите три шага (удаление сносок, шаблон, отбор ссылок) для строки `pairs[10]`. Совпадает ли то, что получилось, с тем, что вы видите в тексте строки?

In [ ]:
section, item = pairs[10]

# ВАШ КОД

#### ✏️💬 Задание 10

Напишите функцию `parse_item(section, item)`, которая возвращает словарь с полями из таблицы выше. Шаги:

1. удалить из `item` сноски и вложенные списки;
2. получить текст и заменить неразрывный пробел;
3. применить шаблон `pattern`; если совпадения нет, поля `year` и `year_end` равны `None`, а в `title` записывается весь текст;
4. отобрать ссылки на произведения: без ссылок на «год в литературе», без внешних ссылок (класс `external`) и без ссылок на другие проекты Википедии, например на Викитеку (класс `extiw`);
5. вернуть словарь.

Проверьте функцию на первых пяти парах, затем примените ко всем парам и превратите результат в таблицу `df`.

In [ ]:
def parse_item(section: str, item) -> dict:
    """Раскладывает строку библиографии на поля."""
    # ВАШ КОД
    ...


for section, item in pairs[:5]:
    print(parse_item(section, item))

In [ ]:
rows = []
for section, item in pairs:
    rows.append(parse_item(section, item))

df = pd.DataFrame(rows)

# Год пришёл строкой; errors="coerce" превращает неподходящие значения в NaN
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["year_end"] = pd.to_numeric(df["year_end"], errors="coerce")

df.head()

In [ ]:
# Строки, которые шаблон не разобрал
df[df["year"].isna()][["section", "text"]]

Несколько строк шаблон не разобрал, например «188х — Красавицы (Из записок врача) (неоконченное)» и «1883 (со 2 июля) — 1885 (по 12 октября) — Осколки московской жизни». Таких строк немного, и на базовом уровне их можно оставить без года: в таблице они видны, и о них нужно сказать при описании набора.

#### ✏️⭐🤖 Задание П1

Попросите ИИ-ассистента написать регулярное выражение, которое разбирает и трудные строки, а обычные строки разбирает так же, как `pattern`.

1. Сформулируйте запрос: приведите ассистенту 4–5 примеров строк, включая трудные, и опишите, какие группы нужны.
2. Проверьте предложенное выражение на всех строках таблицы: сколько строк оно разбирает и не изменился ли результат для строк, которые `pattern` уже разбирал правильно? Проверку оформите кодом, а не на глаз.
3. Запишите в `AI_LOG.md` запрос, ответ и результат проверки: что вы приняли и что исправили.

**Почему здесь можно пользоваться ИИ.** Составлять сложные регулярные выражения долго, и ассистент действительно помогает сэкономить время. Но выражение, которое выглядит правдоподобно, нередко ломается на строках, которых не было в примерах. Поэтому в этом задании главная работа — не получить выражение, а проверить его: именно этот навык нужен, чтобы пользоваться ассистентом в исследовании и не переносить его ошибки в данные.

In [ ]:
new_pattern = r""  # выражение, предложенное ассистентом

# ВАШ КОД: проверка на всех строках df["text"]

---
## Часть 6. Проверка и ответ на вопрос

### 🔁 pandas: группировка и сводные таблицы

| Метод | Что делает |
|---|---|
| `df["столбец"].value_counts()` | сколько раз встречается каждое значение |
| `.sort_index()` | упорядочить результат по значениям, а не по частоте |
| `pd.crosstab(df["a"], df["b"])` | таблица сопряжённости: сколько строк приходится на каждую пару значений |
| `df.groupby("a")["b"].mean()` | среднее значение `b` в каждой группе `a`; для столбца из `True`/`False` — доля `True` |
| `df["столбец"].apply(функция)` | применить функцию к каждому значению столбца |

In [ ]:
# Сколько произведений в каждом разделе
df["section"].value_counts()

In [ ]:
# Рассказы по годам
stories = df[df["section"] == "Рассказы, включая неоконченные"]

stories["year"].value_counts().sort_index()

In [ ]:
# Сколько рассказов, повестей и пьес приходится на каждый год
genres = ["Рассказы, включая неоконченные", "Повести", "Пьесы"]
main = df[df["section"].isin(genres)]

table = pd.crosstab(main["year"], main["section"])
table

Ту же таблицу удобнее рассмотреть на графике. Код ниже строит столбчатую диаграмму с накоплением средствами pandas и matplotlib; разбирать его подробно не нужно.

In [ ]:
ax = table.plot(kind="bar", stacked=True, figsize=(12, 5), width=0.8)
ax.set_xlabel("Год")
ax.set_ylabel("Число произведений")
ax.set_title("Произведения Чехова по годам (по статье «Библиография Антона Чехова»)")
ax.set_xticklabels([int(year) for year in table.index], rotation=90);

#### ✏️📨🚫 Задание 11

1. Прежде чем внимательно смотреть на таблицу и график, запишите гипотезу: как, по-вашему, менялось число рассказов год от года?
2. Найдите три года с наибольшим числом рассказов и год, после которого их число резко падает.
3. По таблице посмотрите, когда появляются повести и пьесы. Совпадает ли резкое падение числа рассказов с появлением повестей?
4. Откройте статью [«Чехов, Антон Павлович»](https://ru.wikipedia.org/wiki/Чехов,_Антон_Павлович) и найдите, что в ней сказано о годах, на которые приходится перелом. Подтверждает ли биография ваше объяснение или предлагает другое?

📨 Отправьте в чат гипотезу, три года, год перелома и объяснение со ссылкой на найденный в биографии факт.

In [ ]:
# ВАШ КОД

### Заметность рассказов

Столбец `has_article` показывает, есть ли у рассказа отдельная статья в Википедии. Можно сравнить два периода: до 1887 года включительно и после. Для этого каждому году сопоставляется подпись периода функцией, а затем считается доля рассказов со статьёй в каждом периоде.

In [ ]:
def period(year: float) -> str:
    """Подпись периода по году."""
    if pd.isna(year):
        return "без года"
    if year <= 1887:
        return "1880–1887"
    return "1888–1905"


# copy: отдельная таблица, чтобы добавление столбца не меняло df
stories = stories.copy()
stories["period"] = stories["year"].apply(period)

stories.groupby("period")["has_article"].agg(["count", "mean"])

#### ✏️📨🚫 Задание 12

Ответьте в ячейке ниже в 4–6 предложениях:

1. Как различаются два периода по числу рассказов и по доле рассказов со статьёй? Что из этого можно сказать о творчестве Чехова?
2. Что эти же цифры говорят о Википедии? Подумайте, кто решает, о каком рассказе писать статью, и почему «заметность» в Википедии не равна значимости рассказа.
3. Насколько можно доверять самому списку? Посмотрите, есть ли у статьи источники, и учтите, что статьи Википедии правятся: номер версии, по которой собран набор, хранится в переменной `revid`.

📨 Отправьте ответ в чат курса.

*Ваш ответ:*

In [ ]:
# Сохраняем таблицу; номер версии статьи — в имени файла
df.to_csv(f"chekhov_bibliography_rev{revid}.csv", index=False, encoding="utf-8")

#### ✏️⭐📨💬 Задание П2. Категория и библиография: два перечня одного корпуса

*Примерное время: 30 минут, несколько запросов.*

**Вопрос.** Совпадают ли два перечня рассказов Чехова в Википедии — категория «Рассказы Антона Чехова» и ссылки в статье-библиографии? Если нет, что объясняет расхождения?

1. Запишите гипотезу: какая доля статей окажется в обоих перечнях?
2. Составьте два множества названий статей: `category_titles` из `stories_category` (часть 2) и `bibliography_titles` из столбца `article` таблицы рассказов (без `None`).
3. Посчитайте пересечение и разности множеств операциями `&` и `-`.
4. Просмотрите по десять названий из каждой разности. Многие расхождения объясняются **перенаправлениями**: в библиографии ссылка ведёт на «В Москве на Трубной площади (рассказ)», а статья называется «В Москве на Трубной площади». Такие ссылки отмечены классом `mw-redirect`. API умеет разрешать перенаправления: запрос с параметрами `action=query`, `titles` (до 50 названий через `|`) и `redirects=1` возвращает в `data["query"]["redirects"]` пары «откуда — куда».
5. Замените в `bibliography_titles` названия-перенаправления их целями и снова посчитайте пересечение и разности.

**Ответ.** Насколько совпадают перечни после разрешения перенаправлений? Какие оставшиеся расхождения говорят о неполноте одного из перечней, а какие — о том, как участники Википедии оформляют статьи и категории?

In [ ]:
category_titles = set(page["title"] for page in stories_category)

# ВАШ КОД

*Гипотеза:*

*Ответ:*

---
## Часть 7. XML

Эта часть, как и часть 3, состоит из двух вариантов. Подраздел 🌱 вводит XML с нуля и показывает тот же ответ API в двух форматах. Подраздел ⭐ разбирает XML полноценно: пространства имён, разметку текстов TEI и корпус пьес DraCor, по которому можно ответить на исследовательские вопросы о драмах Чехова.

### 🌱 XML с нуля

#### Чем XML отличается от HTML

**XML** (eXtensible Markup Language, расширяемый язык разметки) устроен так же, как HTML: теги в угловых скобках, атрибуты, вложенность, дерево. Разница в назначении:

| | HTML | XML |
|---|---|---|
| Для чего | показать страницу человеку | хранить и передавать данные |
| Теги | заданы заранее: `<p>`, `<li>`, `<a>` | придумывает автор формата: `<work>`, `<title>`, `<year>` |
| Что описывают теги | вид: заголовок, список, ссылка | смысл: название, год, автор |
| Ошибки в разметке | браузер старается их исправить | документ с ошибкой считается неправильным |

XML распространён в гуманитарных данных: в нём хранятся тексты в разметке TEI, метаданные библиотек и архивов (протоколы OAI-PMH и SRU), новостные ленты RSS, ответы многих API.

#### Правила хорошо оформленного документа

XML строже HTML. Документ считается **хорошо оформленным** (well-formed), если выполнены пять правил:

1. у документа один **корневой** элемент, внутри которого лежит всё остальное;
2. у каждого открывающего тега есть закрывающий (или тег записан как пустой: `<br/>`);
3. элементы закрываются в обратном порядке вложения;
4. регистр букв в именах тегов важен: `<Title>` и `<title>` — разные теги;
5. значения атрибутов всегда в кавычках.

В начале документа обычно стоит объявление `<?xml version="1.0" encoding="UTF-8"?>`, в котором указаны версия XML и кодировка.

🖐 **Попробуйте.** BeautifulSoup старается прочитать даже неправильный документ, поэтому ошибки удобнее искать строгим парсером из стандартной библиотеки: функция `ET.fromstring()` из модуля `xml.etree.ElementTree` выдаёт ошибку `ParseError` и указывает строку и позицию, где правило нарушено. В каждом из трёх документов нарушено одно правило. Запустите ячейку, по сообщениям найдите ошибки и исправьте документы так, чтобы все три строки вывели «в порядке».

In [ ]:
import xml.etree.ElementTree as ET

broken = [
    "<work><title>Чайка</work>",
    "<work><Title>Чайка</title></work>",
    "<work year=1896><title>Чайка</title></work>",
]

for document in broken:
    try:
        ET.fromstring(document)
        print("в порядке:", document)
    except ET.ParseError as error:
        print("ошибка:", error)

In [ ]:
works_xml = """<?xml version="1.0" encoding="UTF-8"?>
<works author="Антон Чехов">
  <work year="1896" genre="пьеса">
    <title>Чайка</title>
  </work>
  <work year="1888" genre="повесть">
    <title>Степь</title>
  </work>
</works>
"""

# Парсер "xml": учитывает регистр и понимает объявление в начале документа
xml_soup = BeautifulSoup(works_xml, "xml")

# Корневой элемент и его атрибут
print(xml_soup.find("works")["author"])

In [ ]:
# Каждое произведение — элемент work: данные в атрибутах и во вложенном элементе title
for work in xml_soup.find_all("work"):
    print(work["year"], work["genre"], work.find("title").get_text())

#### ✏️🚫 Задание 13

Составьте собственный XML-документ о трёх-четырёх произведениях из вашей области: корневой элемент, у каждого произведения атрибут с годом и вложенные элементы для названия и автора. Разберите документ BeautifulSoup с парсером `"xml"` и превратите его в таблицу pandas со столбцами «год», «название», «автор».

In [ ]:
my_xml = """<?xml version="1.0" encoding="UTF-8"?>

"""

# ВАШ КОД

### XML в ответе API

MediaWiki API умеет отвечать в XML: достаточно указать `format=xml`. Это удобный случай, чтобы сравнить два формата на одних и тех же данных: ниже отправляется знакомый по занятию 2 запрос `categorymembers`.

In [ ]:
params = {
    "action": "query",
    "list": "categorymembers",
    "cmtitle": "Категория:Пьесы Антона Чехова",
    "cmtype": "page",
    "cmlimit": 5,
    "format": "xml",
}

# requests.get, а не get_json: ответ — не JSON, а текст XML
response = requests.get(WIKI_API, params=params, headers=HEADERS, timeout=10)

print(response.text)

Данные те же, что в JSON, но записаны по-другому. Каждая статья — элемент `<cm>` (от categorymember), а её поля (`pageid`, `ns`, `title`) — атрибуты этого элемента. Метка следующей порции лежит в атрибуте `cmcontinue` элемента `<continue>`.

| JSON | XML |
|---|---|
| словарь | элемент |
| ключ и значение | атрибут или вложенный элемент |
| список | несколько одинаковых элементов подряд |

In [ ]:
api_soup = BeautifulSoup(response.text, "xml")

for page in api_soup.find_all("cm"):
    print(page["pageid"], page["title"])

In [ ]:
# Метка следующей порции — атрибут элемента continue
api_soup.find("continue")["cmcontinue"]

#### ✏️🚫 Задание 14

Измените запрос так, чтобы получить все статьи «Категории:Пьесы Антона Чехова» одной порцией (параметр `cmlimit`), разберите ответ в XML и выведите число статей и их названия. Есть ли в ответе элемент `<continue>`?

In [ ]:
# ВАШ КОД

### ⭐ XML для продолжающих: пространства имён и TEI

#### Пространства имён

Раз теги XML придумывает автор формата, в разных форматах одинаковые имена могут значить разное: `<title>` в библиографической записи — название книги, а в описании человека — титул. Чтобы теги разных словарей не путались, в XML есть **пространства имён** (namespaces). Пространство имён объявляется атрибутом `xmlns` и обозначается адресом, который служит уникальным именем словаря (по этому адресу не обязательно что-то открывается):

```xml
<TEI xmlns="http://www.tei-c.org/ns/1.0">          <!-- все теги внутри — из словаря TEI -->
<record xmlns:dc="http://purl.org/dc/elements/1.1/">
  <dc:title>Чайка</dc:title>                         <!-- тег title из словаря Dublin Core -->
</record>
```

Во втором примере у словаря есть **префикс** `dc`, и он ставится перед именем тега через двоеточие. Особый префикс `xml` зарезервирован: атрибуты `xml:id` (идентификатор элемента) и `xml:lang` (язык содержимого) встречаются в документах любых форматов.

BeautifulSoup с парсером `"xml"` по умолчанию находит элементы по имени без префикса: `find("title")` найдёт и `<title>`, и `<dc:title>`. Атрибуты с префиксом читаются под полным именем: `element["xml:id"]`.

🖐 **Попробуйте.** Разберите запись с элементами Dublin Core и выведите название, автора и язык названия (атрибут `xml:lang`).

In [ ]:
record = """<record xmlns:dc="http://purl.org/dc/elements/1.1/">
  <dc:title xml:lang="ru">Вишнёвый сад</dc:title>
  <dc:creator>Чехов, Антон Павлович</dc:creator>
  <dc:date>1904</dc:date>
</record>"""

# ВАШ КОД

#### TEI: разметка гуманитарных текстов

**TEI** (Text Encoding Initiative, Инициатива по кодированию текстов) — международный стандарт разметки текстов для гуманитарных исследований. Работа над ним началась в 1987 году; действующая редакция рекомендаций называется TEI P5 и описывает несколько сотен элементов: для прозы, стихов, драмы, рукописей, писем, словарей.

Главная идея TEI в том, что разметка описывает не вид страницы, а содержание текста, каким его понимает исследователь: имена людей (`<persName>`), названия мест (`<placeName>`), даты (`<date when="1896-10-17">`), правку автора (`<del>`, `<add>`), повреждения рукописи. Поэтому разметка TEI — уже интерпретация: решая, к какому персонажу относится реплика или что считать именем собственным, редактор записывает свои выводы в сам текст, а исследователь, который пользуется размеченным текстом, опирается на эти решения.

Документ TEI состоит из двух частей:

- `<teiHeader>` — заголовок с метаданными: название, автор, источник текста, сведения о редакторах и лицензии, список действующих лиц;
- `<text>` — сам размеченный текст.

Для драмы в TEI есть особые элементы:

| Элемент | Расшифровка | Что размечает |
|---|---|---|
| `<person xml:id="treplev" sex="MALE">` | person, лицо | действующее лицо в списке персонажей заголовка, с идентификатором и полом |
| `<div type="act">` | division, раздел | действие пьесы; `type="scene"` — явление |
| `<sp who="#treplev">` | speech, реплика | реплика; атрибут `who` ссылается на идентификатор персонажа |
| `<speaker>` | говорящий | имя говорящего так, как оно напечатано в тексте |
| `<p>`, `<l>` | paragraph, абзац; line, строка | текст реплики в прозе и в стихах |
| `<stage>` | stage direction, ремарка | авторская ремарка: «(Смотрит на часы.)» |

🖐 **Попробуйте.** Перед тем как загружать целую пьесу, потренируйтесь на одной реплике в разметке TEI. Выведите: идентификатор говорящего без знака `#`, имя из `<speaker>`, текст ремарки и текст реплики без ремарки.

In [ ]:
speech_xml = """<sp who="#treplev">
  <speaker>Треплев</speaker>
  <p>Вот и театр. <stage>(Смотрит на часы.)</stage> Через четверть часа начало.</p>
</sp>"""

# ВАШ КОД

#### Корпус DraCor

**DraCor** (Drama Corpora Project) — проект, который собирает корпуса пьес на разных языках в разметке TEI и даёт к ним открытый API. В русском корпусе RusDraCor больше двухсот пьес, в том числе 14 пьес Чехова; корпус распространяется под лицензией CC0 (общественное достояние). Для каждой пьесы API отдаёт полный текст в TEI, а также готовые сведения о персонажах, по которым можно проверить собственные подсчёты. Документация API — на странице [dracor.org/doc/api](https://dracor.org/doc/api).

Пьеса запрашивается по адресу `https://dracor.org/api/v1/corpora/rus/plays/<имя пьесы>/tei`; «Чайка» называется в корпусе `chekhov-chaika`.

**Чтение по желанию:** Fischer F. et al. Programmable Corpora: Introducing DraCor, an Infrastructure for the Research on European Drama // Proceedings of DH2019. Utrecht, 2019. В статье объясняется, зачем корпусу пьес API и какие вопросы о драме можно задавать к такому корпусу.

In [ ]:
DRACOR_API = "https://dracor.org/api/v1/corpora/rus"

response = requests.get(f"{DRACOR_API}/plays/chekhov-chaika/tei", headers=HEADERS, timeout=30)
response.status_code

In [ ]:
tei_soup = BeautifulSoup(response.text, "xml")

# Корневой элемент и его пространство имён
root_tei = tei_soup.find("TEI")
print(root_tei.get("xmlns"))
print(root_tei.get("xml:lang"))

In [ ]:
# Названия из заголовка: основное, подзаголовок и английское (атрибут xml:lang)
for title in tei_soup.find("titleStmt").find_all("title"):
    print(title.get("xml:lang"), title.get("type"), title.get_text())

In [ ]:
# Список действующих лиц: идентификатор, пол, имя
persons = []
for person in tei_soup.find_all("person"):
    persons.append({
        "id": person["xml:id"],
        "sex": person.get("sex"),
        "name": person.find("persName").get_text(),
    })

pd.DataFrame(persons)

In [ ]:
# Первая реплика пьесы целиком
first_speech = tei_soup.find("sp")
print(first_speech)

In [ ]:
# Сколько всего реплик в пьесе
len(tei_soup.find_all("sp"))

🖐 **Попробуйте.** Посчитайте, сколько в «Чайке» действий (`div` с атрибутом `type="act"`; поиск по атрибуту записывается как `find_all("div", attrs={"type": "act"})`) и ремарок (`stage`), и выведите атрибут `who` десятой реплики.

In [ ]:
# ВАШ КОД

#### ✏️⭐📨💬 Задание П3. Кто главный в «Чайке»

*Примерное время: 30 минут. Один запрос уже сделан выше.*

**Вопрос.** Кого из персонажей «Чайки» можно считать главным, если судить по тексту пьесы, и зависит ли ответ от того, как измерять «главность»?

1. Запишите гипотезу: кто, по-вашему, говорит больше всех?
2. Посчитайте для каждого персонажа число реплик: пройдите по всем элементам `sp` и возьмите атрибут `who`. Учтите, что в `who` может стоять несколько идентификаторов через пробел (реплика нескольких персонажей), и уберите знак `#` в начале.
3. Посчитайте для каждого персонажа число слов в репликах: текст элементов `p` и `l` внутри `sp`, без ремарок `stage`. Ремарки удобно удалить методом `.decompose()` до подсчёта, а слова посчитать как `len(text.split())`.
4. Сведите оба показателя в таблицу с именами персонажей и сравните первые места.
5. Проверьте свои подсчёты по API: адрес `f"{DRACOR_API}/plays/chekhov-chaika/characters"` возвращает список персонажей с полями `numOfSpeechActs` (число реплик) и `numOfWords` (число слов). Насколько ваши числа совпадают с этими и чем можно объяснить расхождения?

**Ответ.** Кто главный по числу реплик и кто — по числу слов? Что каждый из двух показателей говорит о роли персонажа и что в нём зависит от решений редакторов корпуса (например, как разметить общую реплику или ремарку внутри реплики)?

In [ ]:
speech_acts = {}
words = {}

# ВАШ КОД

*Гипотеза:*

*Ответ:*

#### ✏️⭐📨💬 Задание П4. Мужские и женские голоса в больших пьесах

*Примерное время: 40 минут, четыре запроса.*

**Вопрос.** Какая доля текста в четырёх больших пьесах Чехова («Чайка», «Дядя Ваня», «Три сестры», «Вишнёвый сад») принадлежит женским персонажам и одинакова ли она в разных пьесах?

1. Запишите гипотезу.
2. Напишите функцию `speech_by_sex(play_name)`, которая загружает TEI пьесы, строит словарь «идентификатор → пол» по элементам `person` и считает число слов мужских и женских персонажей тем же способом, что в задании П3. Функция возвращает словарь с названием пьесы, числом слов каждой группы и долей женских.
3. Вызовите функцию для `chekhov-chaika`, `chekhov-djadja-vanja`, `chekhov-tri-sestry`, `chekhov-vishnevyi-sad` с паузой в 1 секунду и сведите результаты в таблицу.
4. Проверьте результат через API: поля `sex` и `numOfWords` в ответе `characters` позволяют посчитать ту же долю без разбора TEI.

**Ответ.** Одинакова ли доля женской речи в четырёх пьесах? Что результат говорит о драматургии Чехова, а что зависит от устройства корпуса: кто и как присвоил персонажам пол, как учтены групповые персонажи (поле `isGroup`)?

In [ ]:
def speech_by_sex(play_name: str) -> dict:
    """Число слов мужских и женских персонажей пьесы по разметке TEI."""
    # ВАШ КОД
    return {}


# ВАШ КОД: четыре пьесы и таблица

*Гипотеза:*

*Ответ:*

#### ✏️⭐📨🤖 Задание П5. Собственный вопрос к корпусу

*Сложность: высокая, задача творческая.*

Сформулируйте собственный исследовательский вопрос к пьесам Чехова или к другим пьесам корпуса RusDraCor (список пьес можно получить запросом `get_json(DRACOR_API, {})["plays"]`). Ответьте на него в обычном порядке: гипотеза, расчёт по разметке TEI, ответ с учётом того, как устроен корпус. В этом задании можно пользоваться ИИ-ассистентом, например чтобы разобраться в незнакомых элементах TEI; обращения записываются в `AI_LOG.md`, а каждое утверждение ассистента о разметке проверяется по самому документу.

**Примеры вопросов:**

- Как меняется число реплик главных персонажей от действия к действию (`div type="act"`)?
- Какую долю текста «Вишнёвого сада» составляют ремарки и отличается ли она от ранних одноактных пьес Чехова?
- Сколько персонажей появляется только в одном действии?

📨 Отправьте в чат вопрос и ответ.

In [ ]:
# ВАШ КОД

*Вопрос:*

*Гипотеза:*

*Ответ:*

---
## Где потренироваться

Если HTML, CSS-селекторы и XML даются с трудом, их стоит отработать в интерактивных курсах, где код пишется прямо в браузере и сразу проверяется.

| Курс | Язык | Что там |
|---|---|---|
| [MDN: Начало работы с HTML](https://developer.mozilla.org/ru/docs/Learn_web_development/Core/Structuring_content/Basic_HTML_syntax) | русский | подробное введение в HTML от разработчиков браузера Firefox, с примерами, которые можно править |
| [HTML Academy](https://htmlacademy.ru/courses) | русский | интерактивные курсы по HTML и CSS для начинающих; условия доступа к курсам указаны на сайте |
| [W3Schools: HTML Tutorial](https://www.w3schools.com/html/) | английский | короткие уроки по каждому тегу с редактором «Try it Yourself» и упражнениями |
| [CSS Diner](https://flukeout.github.io/) | английский | игра из 32 уровней, в которой нужно подобрать CSS-селектор к элементам на «столе» |
| [W3Schools: XML Tutorial](https://www.w3schools.com/xml/) | английский | введение в XML: правила оформления, атрибуты, пространства имён |
| [TEI Guidelines](https://tei-c.org/guidelines/) | английский | рекомендации TEI P5; для начала полезны главы о заголовке TEI и о разметке драмы |

---
## Итоги

На занятии пройден путь от страницы, написанной для человека, к таблице:

`вопрос` → `где лежат данные` → `страница через API` → `дерево HTML` → `поиск элементов` → `строка → поля` → `таблица` → `проверка` → `интерпретация с учётом источника` → `CSV`

Технически новыми были HTML и его устройство (теги, атрибуты, вложенность), инструменты разработчика в браузере, библиотека BeautifulSoup, CSS-селекторы и XML. Не менее важно, что разбор страницы требует решений: что считать строкой списка, какую ссылку считать ссылкой на произведение, что делать со строками, которые не подходят под шаблон. Эти решения влияют на итоговые цифры и должны быть описаны вместе с набором данных. Разметка TEI показывает ту же мысль с другой стороны: размеченный текст содержит решения редактора, и исследователь, который считает реплики или слова, опирается на эти решения.

На занятии 4 тот же приём переносится на сайт, у которого нет API: страниц будет много, и понадобятся обход по ссылкам, паузы и проверка правил сайта.

### Документация

- MediaWiki API, разбор страниц: https://www.mediawiki.org/wiki/API:Parsing_wikitext
- Beautiful Soup: https://www.crummy.com/software/BeautifulSoup/bs4/doc/
- DraCor API: https://dracor.org/doc/api
- TEI Guidelines: https://tei-c.org/guidelines/